# Ablation: Embedding Models

Compares three retrieval architectures:
- **Sentence-BERT** (two-tower): encodes query and passage independently, fast retrieval via FAISS
- **Cross-Encoder** (KNRM-style): jointly encodes query+passage pairs, slower but stronger reranking
- **ColBERT** (late interaction): per-token embeddings with MaxSim scoring


In [1]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModel

from loader import load_data
from retrievers.bm25 import BM25Retriever
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

N = 10_000
TOP_K = 10
ds = load_data(n=N)
passages_text, queries = [], []
for example in ds:
    queries.append(example["query"])
    for p in example["passages"]["passage_text"]:
        passages_text.append(p)
print(f"Passages: {len(passages_text)}, Queries: {len(queries)}")

/home/skimura/Projects/ics624_document_retrieval_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


Passages: 99736, Queries: 10000


## 1. Sentence-BERT Variants (Two-Tower)

All three models use the same FAISS IndexFlatIP retrieval pipeline — only the encoder changes.

In [2]:
def evaluate_sbert(model_name, passages, ds, queries, top_k=TOP_K, batch_size=64):
    model = SentenceTransformer(model_name, device=DEVICE)
    print(f"Encoding {len(passages)} passages with {model_name}...")
    emb_file = f"sbert_{model_name.replace('/', '_')}.npy"
    if os.path.exists(emb_file) and np.load(emb_file, mmap_mode='r').shape[0] == len(passages):
        doc_embs = np.load(emb_file).astype(np.float32)
    else:
        doc_embs = model.encode(passages, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
        np.save(emb_file, doc_embs)
    norms = np.linalg.norm(doc_embs, axis=1, keepdims=True)
    doc_embs /= norms
    index = faiss.IndexFlatIP(doc_embs.shape[1])
    index.add(doc_embs)

    class _Retriever:
        def query(self, q):
            q_emb = model.encode([q], convert_to_numpy=True).astype(np.float32)
            q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
            _, idx = index.search(q_emb, top_k)
            return idx[0].tolist()
        def query_batch(self, qs):
            q_embs = model.encode(qs, batch_size=batch_size, convert_to_numpy=True).astype(np.float32)
            q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
            _, idxs = index.search(q_embs, top_k)
            return idxs.tolist()

    retriever = _Retriever()
    mrr = mrr_at_10(retriever, ds)
    avg_t = measure_retrieval_time(retriever, queries)

    # Free GPU memory before loading the next model
    del model, doc_embs, index
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return mrr, avg_t

In [3]:
sbert_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1",
]

results = []
for model_name in sbert_models:
    print(f"\n--- {model_name} ---")
    mrr, avg_t = evaluate_sbert(model_name, passages_text, ds, queries)
    results.append({"Model": model_name, "Architecture": "Two-Tower (SBERT)", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_t * 1000, 3)})
    print(f"MRR@10: {mrr:.4f}, Avg ms/query: {avg_t*1000:.3f}")


--- all-MiniLM-L6-v2 ---


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11088.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 99736 passages with all-MiniLM-L6-v2...
Running query_batch on 100 queries...
MRR@10: 0.6012, Avg ms/query: 9.831

--- all-mpnet-base-v2 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10822.54it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 99736 passages with all-mpnet-base-v2...


...

Running query_batch on 100 queries...
MRR@10: 0.6324, Avg ms/query: 20.714

--- multi-qa-MiniLM-L6-cos-v1 ---


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11925.50it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 99736 passages with multi-qa-MiniLM-L6-cos-v1...


Batches: 100%|██████████| 1559/1559 [00:36<00:00, 42.44it/s]


Running query_batch on 100 queries...
MRR@10: 0.6024, Avg ms/query: 11.228


## 2. Cross-Encoder Reranking (KNRM-style)

A cross-encoder jointly encodes the query and each candidate passage, enabling token-level interaction.
Because it cannot pre-index passages, it runs as a two-stage pipeline: BM25 retrieves candidates, the cross-encoder reranks them.

In [4]:
class CrossEncoderRetriever:
    def __init__(self, model_name, candidate_retriever, top_k=TOP_K):
        self.model = CrossEncoder(model_name, device=DEVICE)
        self.candidate_retriever = candidate_retriever
        self.top_k = top_k
        self.passages = None

    def fit(self, passages):
        self.passages = passages
        self.candidate_retriever.fit(passages)

    def query(self, query):
        candidates = self.candidate_retriever.query(query)
        pairs = [(query, self.passages[i]) for i in candidates]
        scores = self.model.predict(pairs)
        ranked = sorted(zip(candidates, scores.tolist()), key=lambda x: x[1], reverse=True)
        return [idx for idx, _ in ranked[:self.top_k]]

In [5]:
ce_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ce_retriever = CrossEncoderRetriever(
    model_name=ce_model_name,
    candidate_retriever=BM25Retriever(top_k=100),
    top_k=TOP_K,
)
ce_retriever.fit(passages_text)

print("Running cross-encoder MRR evaluation (slow — no batching)...")
mrr = mrr_at_10(ce_retriever, ds, max_queries=500)
avg_t = measure_retrieval_time(ce_retriever, queries)
results.append({"Model": ce_model_name, "Architecture": "Cross-Encoder (KNRM-style)", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_t * 1000, 3)})
print(f"MRR@10: {mrr:.4f}, Avg ms/query: {avg_t*1000:.3f}")
del ce_retriever; gc.collect()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10961.54it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running cross-encoder MRR evaluation (slow — no batching)...
Processed 100 queries...
Processed 200 queries...
Processed 300 queries...
Processed 400 queries...
Processed 500 queries...
MRR@10: 0.6208, Avg ms/query: 65.383


2059

## 3. Simplified ColBERT (Late Interaction)

ColBERT encodes query and passage into per-token embeddings, then scores via MaxSim:
`score(q,d) = Σ_i max_j cos(q_i, d_j)`

Full ColBERT pre-indexes all passage token embeddings (~150GB for MS MARCO).
This simplified version retrieves BM25 candidates and reranks them with MaxSim on-the-fly.

In [6]:
class ColBERTRetriever:
    def __init__(self, model_name, candidate_retriever, top_k=TOP_K):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
        self.candidate_retriever = candidate_retriever
        self.top_k = top_k
        self.passages = None

    def fit(self, passages):
        self.passages = passages
        self.candidate_retriever.fit(passages)

    def _token_emb(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            out = self.model(**inputs)
        emb = out.last_hidden_state[0]  # (seq_len, dim)
        return torch.nn.functional.normalize(emb, dim=-1)

    def _maxsim(self, q_emb, d_emb):
        sim = torch.matmul(q_emb, d_emb.T)  # (q_len, d_len)
        return sim.max(dim=1).values.sum().item()

    def query(self, query):
        candidates = self.candidate_retriever.query(query)
        q_emb = self._token_emb(query)
        scores = [(idx, self._maxsim(q_emb, self._token_emb(self.passages[idx]))) for idx in candidates]
        return [idx for idx, _ in sorted(scores, key=lambda x: x[1], reverse=True)[:self.top_k]]

In [7]:
colbert_retriever = ColBERTRetriever(
    model_name="bert-base-uncased",
    candidate_retriever=BM25Retriever(top_k=100),
    top_k=TOP_K,
)
colbert_retriever.fit(passages_text)

print("Running ColBERT MRR evaluation (slow — per-token encoding per candidate)...")
mrr = mrr_at_10(colbert_retriever, ds, max_queries=200)
avg_t = measure_retrieval_time(colbert_retriever, queries)
results.append({"Model": "bert-base-uncased", "Architecture": "ColBERT (Late Interaction)", "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_t * 1000, 3)})
print(f"MRR@10: {mrr:.4f}, Avg ms/query: {avg_t*1000:.3f}")
del colbert_retriever; gc.collect()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11629.25it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running ColBERT MRR evaluation (slow — per-token encoding per candidate)...
Processed 100 queries...
Processed 200 queries...
MRR@10: 0.2868, Avg ms/query: 465.693


29

## Results

In [8]:
df = pd.DataFrame(results)
df.to_csv("ablation_embedding_models.csv", index=False)
print(df.to_string(index=False))

                               Model               Architecture  MRR@10  Avg ms/query
                    all-MiniLM-L6-v2          Two-Tower (SBERT)  0.6012         9.831
                   all-mpnet-base-v2          Two-Tower (SBERT)  0.6324        20.714
           multi-qa-MiniLM-L6-cos-v1          Two-Tower (SBERT)  0.6024        11.228
cross-encoder/ms-marco-MiniLM-L-6-v2 Cross-Encoder (KNRM-style)  0.6208        65.383
                   bert-base-uncased ColBERT (Late Interaction)  0.2868       465.693
